## metodo_buffer

In [1]:
import geopandas as gpd
import pandas as pd

# ==========================================
# 1. Configuración de rutas
# ==========================================
archivo_puntos = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_centroides.gpkg"
archivo_mst = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_mst_por_ubigeo.gpkg"
archivo_salida = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\metodos_poligonos\poligonos_buffer.gpkg"

atributo_clave = "UBIGEO_CCPP_CONFIRMADO"
distancia_buffer = 50  # Distancia en metros para "inflar" la geometría

try:
    print("Cargando capas...")
    gdf_puntos = gpd.read_file(archivo_puntos)
    gdf_mst = gpd.read_file(archivo_mst)

    # Filtrar puntos (los que no son 0)
    gdf_puntos = gdf_puntos[(gdf_puntos[atributo_clave] != 0) & (gdf_puntos[atributo_clave] != '0')]

    print(f"Aplicando un buffer de {distancia_buffer} metros...")
    # Creamos copias y aplicamos el buffer a ambas capas
    puntos_buf = gdf_puntos.copy()
    puntos_buf['geometry'] = puntos_buf.geometry.buffer(distancia_buffer)

    mst_buf = gdf_mst.copy()
    mst_buf['geometry'] = mst_buf.geometry.buffer(distancia_buffer)

    print("Uniendo y disolviendo por UBIGEO...")
    # Juntamos ambas capas en una sola tabla usando pandas
    # Mantenemos solo el atributo clave y la geometría para evitar conflictos de columnas
    capa_unida = pd.concat([
        puntos_buf[[atributo_clave, 'geometry']], 
        mst_buf[[atributo_clave, 'geometry']]
    ], ignore_index=True)

    # Volvemos a convertirlo en GeoDataFrame
    gdf_unido = gpd.GeoDataFrame(capa_unida, geometry='geometry', crs=gdf_puntos.crs)

    # DISSOLVE: Magia pura. Fusiona todos los polígonos que toquen el mismo UBIGEO
    gdf_final = gdf_unido.dissolve(by=atributo_clave).reset_index()

    print("Guardando capa de Polígonos (Buffer)...")
    gdf_final.to_file(archivo_salida, driver="GPKG")
    print("¡Método de Buffer completado con éxito!")

except Exception as e:
    print(f"Ocurrió un error: {e}")

Cargando capas...
Aplicando un buffer de 50 metros...
Uniendo y disolviendo por UBIGEO...
Guardando capa de Polígonos (Buffer)...
¡Método de Buffer completado con éxito!


## metodo_voronoi

In [6]:
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
from shapely.ops import voronoi_diagram, unary_union
import numpy as np
import warnings

# Ignorar advertencias menores
warnings.filterwarnings("ignore")

# ==========================================
# 1. Configuración de rutas
# ==========================================
archivo_puntos = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_centroides.gpkg"
archivo_tif = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CO_2510221339013\VOL_PER1_ORT_001_002705\IMG_PER1_ORT_PMS_002705\IMG_PER1_20210706153514_ORT_PMS_002705.TIF"

#archivo_tif = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\CO_2510221339013\VOL_PER1_ORT_001_002705\IMG_PER1_ORT_PMS_002705\IMG_PER1_20210706153514_ORT_PMS_002705.TIF" # <-- ACTUALIZA ESTA RUTA

archivo_salida = r"D:\Imagen_Sat_F\CAJAMARCA DIEGO\metodos_poligonos\voronoi_recortado_nodata.gpkg"
atributo_clave = "UBIGEO_CCPP_CONFIRMADO"

try:
    # ==========================================
    # 2. Extraer la HUELLA EXACTA (Sin NoData) del TIF
    # ==========================================
    print("Analizando los píxeles de la imagen satelital para extraer la huella real...")
    with rasterio.open(archivo_tif) as src:
        # Leemos toda la imagen
        arr = src.read()
        
        # Aplicamos tu lógica: Creamos una máscara donde NO todas las bandas sean 0
        # axis=0 compara a través de las bandas de la imagen
        mascara_valida = np.any(arr != 0, axis=0).astype('uint8')
        
        print("Convirtiendo la huella de píxeles a un polígono vectorial...")
        # rasterio.features.shapes convierte los píxeles válidos (1) en geometrías de polígono
        generador_geometrias = shapes(mascara_valida, mask=mascara_valida.astype(bool), transform=src.transform)
        
        # Extraemos las geometrías y las convertimos al formato de Shapely
        poligonos_huella = [shape(geom) for geom, valor in generador_geometrias if valor == 1]
        
        # Fusionamos todos los polígonos resultantes en uno solo (por si la imagen tiene huecos)
        huella_total = unary_union(poligonos_huella)
        crs_tif = src.crs

    # Creamos un GeoDataFrame con esta huella exacta
    gdf_huella_tif = gpd.GeoDataFrame({'geometry': [huella_total]}, crs=crs_tif)

    # ==========================================
    # 3. Cargar y preparar Puntos
    # ==========================================
    print("Cargando capa de puntos...")
    gdf_puntos = gpd.read_file(archivo_puntos)
    gdf_puntos = gdf_puntos[(gdf_puntos[atributo_clave] != 0) & (gdf_puntos[atributo_clave] != '0')]
    gdf_puntos = gdf_puntos.drop_duplicates(subset=['geometry']).reset_index(drop=True)

    # Verificar que usen el mismo CRS
    if gdf_puntos.crs != gdf_huella_tif.crs:
        print("Proyectando la huella de la imagen al mismo sistema de los puntos...")
        gdf_huella_tif = gdf_huella_tif.to_crs(gdf_puntos.crs)

    # ==========================================
    # 4. Calcular Voronoi
    # ==========================================
    print("Calculando red de Voronoi...")
    # Usamos un sobre (envelope) holgado para el cálculo inicial
    limite_exterior = gdf_puntos.unary_union.convex_hull.buffer(2000)
    voronoi_geoms = voronoi_diagram(gdf_puntos.unary_union, envelope=limite_exterior)
    gdf_voronoi = gpd.GeoDataFrame(geometry=[geom for geom in voronoi_geoms.geoms], crs=gdf_puntos.crs)

    # ==========================================
    # 5. Asignar UBIGEO y Disolver
    # ==========================================
    print("Asignando UBIGEO a cada polígono...")
    # Usamos un punto interno representativo para el join espacial
    gdf_puntos['rep_point'] = gdf_puntos.geometry
    gdf_voronoi = gpd.sjoin(gdf_voronoi, gdf_puntos[[atributo_clave, 'geometry']], how='left', predicate='intersects')

    print("Disolviendo límites internos por UBIGEO...")
    gdf_final = gdf_voronoi.dissolve(by=atributo_clave).reset_index()
    if 'index_right' in gdf_final.columns:
        gdf_final = gdf_final.drop(columns=['index_right'])

    # ==========================================
    # 6. RECORTAR (CLIP) CON LA HUELLA REAL (SIN NODATA)
    # ==========================================
    print("Recortando los polígonos al área exacta con datos de la imagen satelital...")
    gdf_recortado = gpd.clip(gdf_final, gdf_huella_tif)

    # ==========================================
    # 7. Guardar Resultado
    # ==========================================
    print("Guardando capa final...")
    gdf_recortado.to_file(archivo_salida, driver="GPKG")
    print("¡Proceso completado! Los bordes ahora calzarán perfectamente con los píxeles útiles de tu imagen.")

except Exception as e:
    print(f"Ocurrió un error: {e}")

Analizando los píxeles de la imagen satelital para extraer la huella real...
Convirtiendo la huella de píxeles a un polígono vectorial...
Cargando capa de puntos...
Calculando red de Voronoi...
Asignando UBIGEO a cada polígono...
Disolviendo límites internos por UBIGEO...
Recortando los polígonos al área exacta con datos de la imagen satelital...
Guardando capa final...
¡Proceso completado! Los bordes ahora calzarán perfectamente con los píxeles útiles de tu imagen.


## UNION DE CARRETERAS 

In [ ]:
# ==========================================
# 1. Configuración de rutas
# ==========================================
archivo_puntos = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_centroides.gpkg"
archivo_vias = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\tu_capa_de_carreteras.gpkg" # <-- REVISA ESTA RUTA
archivo_salida = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\mst_red_vial.gpkg"
atributo_clave = "UBIGEO_CCPP_CONFIRMADO"

In [15]:
import geopandas as gpd
import networkx as nx
import momepy
from scipy.spatial import cKDTree
import numpy as np
from shapely.geometry import LineString
import warnings
import sys  # <-- NUEVO: Para detener el script si la capa es incorrecta

warnings.filterwarnings("ignore")

# ==========================================
# 1. Configuración de rutas
# ==========================================
archivo_puntos = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_centroides.gpkg"
archivo_vias = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\capa_carreteras_sierra.gpkg" # <-- REVISA ESTA RUTA
archivo_salida = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\mst_red_vial.gpkg"
atributo_clave = "UBIGEO_CCPP_CONFIRMADO"

try:
    # ==========================================
    # 2. Cargar Capas
    # ==========================================
    print("Cargando viviendas y red de carreteras...")
    gdf_puntos = gpd.read_file(archivo_puntos)
    gdf_vias = gpd.read_file(archivo_vias)
    
    # ==========================================
    # NUEVO: DIAGNÓSTICO DE GEOMETRÍA DE LA CAPA VIAL
    # ==========================================
    print("\n--- DIAGNÓSTICO DE LA CAPA DE CARRETERAS ---")
    # Obtener los tipos de geometría únicos en tu archivo
    tipos_geometria = gdf_vias.geometry.type.unique()
    print(f"Tipos de geometría encontrados en tu archivo: {tipos_geometria}")
    
    # Comprobar si existe al menos un tipo de línea
    tiene_lineas = any(tipo in ['LineString', 'MultiLineString'] for tipo in tipos_geometria)
    
    if not tiene_lineas:
        print("\n[!] ERROR CRÍTICO: El archivo que asignaste a 'archivo_vias' NO contiene líneas.")
        print("    El algoritmo de redes viales necesita calles (LineString).")
        print("    Por favor, revisa la ruta y asegúrate de cargar el archivo correcto.")
        sys.exit()  # Detenemos el script aquí mismo para evitar el colapso
    else:
        print("¡Excelente! El archivo contiene líneas. Continuando con el proceso...\n")
    # ==========================================

    # Filtrar casas válidas
    gdf_puntos = gdf_puntos[(gdf_puntos[atributo_clave] != 0) & (gdf_puntos[atributo_clave] != '0')]
    
    # Validar Sistemas de Coordenadas (EPSG:32718)
    if gdf_puntos.crs != gdf_vias.crs:
        print("Alineando sistemas de coordenadas...")
        gdf_vias = gdf_vias.to_crs(gdf_puntos.crs)

    # ==========================================
    # 3. Limpieza de geometrías inválidas en vías
    # ==========================================
    print("Limpiando la capa de carreteras (filtrando basura)...")
    vias_originales = len(gdf_vias)
    
    gdf_vias = gdf_vias.dropna(subset=['geometry'])
    tipos_validos = ['LineString', 'MultiLineString']
    gdf_vias = gdf_vias[gdf_vias.geometry.type.isin(tipos_validos)]
    gdf_vias = gdf_vias.explode(ignore_index=True)
    
    print(f"Se descartaron geometrías inválidas. Vías listas para procesar: {len(gdf_vias)} (Originales: {vias_originales})")

    gdf_vias['longitud_m'] = gdf_vias.geometry.length

    # ==========================================
    # 4. Convertir Carreteras en Grafo Matemático
    # ==========================================
    print("Convirtiendo carreteras a grafo navegable...")
    G_vias = momepy.gdf_to_nx(gdf_vias, approach='primal', length='longitud_m')
    
    # ==========================================
    # 5. "Ajustar" (Snap) las casas a la carretera más cercana
    # ==========================================
    print("Buscando el punto de acceso vial más cercano para cada vivienda...")
    nodos_viales = list(G_vias.nodes)
    coords_viales = np.array(nodos_viales)
    
    arbol_busqueda = cKDTree(coords_viales)
    coords_casas = np.array([(geom.x, geom.y) for geom in gdf_puntos.geometry])
    
    distancias, indices_cercanos = arbol_busqueda.query(coords_casas)
    gdf_puntos['nodo_red'] = [nodos_viales[i] for i in indices_cercanos]

    # ==========================================
    # 6. Calcular la Red MST (Steiner Tree) por UBIGEO
    # ==========================================
    print("Calculando rutas óptimas a través de la red vial por UBIGEO...")
    grupos_ubigeo = gdf_puntos.groupby(atributo_clave)
    
    lineas_mst = []
    ubigeos_mst = []
    
    for ubigeo, grupo in grupos_ubigeo:
        nodos_terminales = grupo['nodo_red'].unique().tolist()
        
        if len(nodos_terminales) < 2:
            continue
            
        try:
            arbol_optimo = nx.approximation.steiner_tree(G_vias, nodos_terminales, weight='longitud_m')
            
            for u, v, datos in arbol_optimo.edges(data=True):
                if 'geometry' in datos:
                    lineas_mst.append(datos['geometry'])
                    ubigeos_mst.append(ubigeo)
                else:
                    lineas_mst.append(LineString([u, v]))
                    ubigeos_mst.append(ubigeo)
                    
        except nx.NetworkXNoPath:
            pass # Ignoramos silenciosamente si no hay conexión

    # ==========================================
    # 7. Guardar Resultados
    # ==========================================
    print("Guardando el diseño de red final...")
    gdf_resultado = gpd.GeoDataFrame({
        atributo_clave: ubigeos_mst,
        "geometry": lineas_mst
    }, crs=gdf_puntos.crs)
    
    if not gdf_resultado.empty:
        gdf_resultado = gdf_resultado.dissolve(by=atributo_clave).reset_index()
        gdf_resultado.to_file(archivo_salida, driver="GPKG")
        print("¡Proceso completado! Tienes tu diseño de red basado en carreteras reales.")
    else:
        print("No se generó ninguna red. Verifica que los puntos estén cerca de las vías.")

except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

Cargando viviendas y red de carreteras...

--- DIAGNÓSTICO DE LA CAPA DE CARRETERAS ---
Tipos de geometría encontrados en tu archivo: ['Polygon']

[!] ERROR CRÍTICO: El archivo que asignaste a 'archivo_vias' NO contiene líneas.
    El algoritmo de redes viales necesita calles (LineString).
    Por favor, revisa la ruta y asegúrate de cargar el archivo correcto.


SystemExit: 